# Topic: ML: K-Nearest Neighbors (KNN) & Distance Metrics

## Definition (30-second explanation)
K-Nearest Neighbors (KNN) is a simple, non-parametric, lazy learning algorithm used for both classification and regression. It predicts the label or value of a new data point by finding the 'K' closest data points in the training set and taking a majority vote (for classification) or an average (for regression), using a specific distance metric to define "closeness."

## Why Interviewers Ask This
*   **Fundamental algorithm:** It tests your understanding of instance-based learning vs. model-based learning.
*   **Distance Metrics:** It's the standard entry point to discuss how to mathematically define similarity (Euclidean vs. Cosine vs. Manhattan).
*   **Curse of Dimensionality:** KNN is the classic algorithm used to test if you understand why distance breaks down in high-dimensional spaces.
*   **Production Awareness:** Interviewers want to see if you know *not* to use basic KNN for low-latency, large-scale production scoring.

## Core Concepts
*   **Lazy Learning:** No explicit training phase; the model *is* the training data. Computation happens entirely at inference time.
*   **Choice of K:** A small K (e.g., K=1) has low bias but high variance (overfitting, sensitive to noise). A large K has high bias but low variance (underfitting).
*   **Distance Metrics:** 
    *   *Euclidean:* Standard straight-line distance (L2 norm). Sensitive to scale.
    *   *Manhattan:* Grid-like path (L1 norm). Often better for high dimensions or categorical grids.
    *   *Cosine:* Measures the angle between vectors, ignoring magnitude. Great for text/TF-IDF.
    *   *Mahalanobis:* Accounts for covariance and scaling. Useful when features are correlated.
*   **Curse of Dimensionality:** In high dimensions, the distance between the nearest and farthest neighbors becomes almost equal, rendering "closeness" meaningless.

## When to Use
*   When a simple baseline model is needed quickly.
*   When the decision boundary is highly irregular and non-linear.
*   When interpretability is important (you can easily show *why* a prediction was made by showing the neighbors).
*   When the dataset is relatively small and feature dimensions are low.

## Advantages
*   No assumptions about the underlying data distribution (non-parametric).
*   Incredibly simple to implement and understand.
*   Naturally handles multi-class classification out of the box.
*   Can adapt immediately to new training data (just add it to the dataset).

## Limitations
*   **Computationally expensive at inference:** Must calculate distance to every single training point for every prediction $O(N \times D)$.
*   **Memory intensive:** Requires storing the entire dataset in memory.
*   Requires rigorous feature scaling (normalization/standardization); otherwise, large-scale features dominate the distance calculation.
*   Fails dramatically in high-dimensional spaces.

## Common Comparisons
*   **KNN vs. Decision Trees:** Trees are eager learners, handle unscaled data well, and do feature selection implicitly. KNN is lazy, needs scaled data, and uses all features equally (unless weighted).
*   **KNN vs. Logistic Regression:** Logistic regression assumes a linear decision boundary and outputs probabilities. KNN can model highly non-linear boundaries.

## Common Interview Traps
*   **Forgetting to scale data:** The #1 trap. Always mention standardizing features before applying KNN.
*   **Ignoring inference latency:** Suggesting KNN for a real-time web application with millions of users without mentioning approximate nearest neighbors (ANN, e.g., FAISS).
*   **Misunderstanding K=Even:** For binary classification, choosing an even K can result in ties. (Rule of thumb: use an odd K for binary classification).

## Python / SQL Syntax (if applicable)
```python
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Always scale data first!
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit and predict
knn = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2) # p=2 is Euclidean
knn.fit(X_scaled, y)
predictions = knn.predict(X_new_scaled)
```

## Important Formula (if applicable)
*   **Euclidean Distance (L2):** $$d(\mathbf{p}, \mathbf{q}) = \sqrt{\sum_{i=1}^{n} (q_i - p_i)^2}$$
*   **Manhattan Distance (L1):** $$d(\mathbf{p}, \mathbf{q}) = \sum_{i=1}^{n} |q_i - p_i|$$
*   **Cosine Similarity:** $$S_C(\mathbf{A}, \mathbf{B}) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\|\|\mathbf{B}\|}$$

## 45-Second Interview Answer
"KNN is a non-parametric, lazy learning algorithm that classifies new data points based on the majority class of their 'K' closest neighbors in the feature space. Because it calculates distances—typically Euclidean or Manhattan—feature scaling is absolutely mandatory before using it. Its primary advantage is its simplicity and ability to model highly complex, non-linear boundaries. However, it suffers from high inference latency, high memory requirements, and the curse of dimensionality, making it unsuitable for very large or high-dimensional datasets unless we use Approximate Nearest Neighbor techniques like FAISS."

## Practice Questions:

### Q1:
You are building a content-based recommender system for a news application. You have converted the articles into TF-IDF vectors (high-dimensional text embeddings). You need to write a simple script to find the 2 most similar articles to a given target article using K-Nearest Neighbors.

**The Question:**
Which distance/similarity metric is best suited for this TF-IDF text data and why? Write a short Python snippet using scikit-learn to instantiate and fit a Nearest Neighbors model to retrieve the 2 nearest neighbors for 'Doc_A'.

In [18]:
# Data:
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# Mock TF-IDF DataFrame representing 4 documents (A, B, C, D) and 4 vocabulary words
data = {
    'doc_id': ['Doc_A', 'Doc_B', 'Doc_C', 'Doc_D'],
    'word_AI': [0.8, 0.1, 0.0, 0.9],
    'word_ML': [0.7, 0.0, 0.1, 0.8],
    'word_Sports': [0.0, 0.9, 0.8, 0.1],
    'word_Food': [0.0, 0.1, 0.9, 0.0]
}
df = pd.DataFrame(data)

# Features only
X = df.drop('doc_id', axis=1)

**Answer:**
*   **Metric:** Cosine Similarity (or Cosine Distance).
*   **Why:** Cosine similarity measures the angle between two vectors, completely ignoring their magnitude. In text data, magnitude is largely driven by document length. We want a short article about "Machine Learning" and a long article about "Machine Learning" to be considered highly similar, which Euclidean distance would fail to do.

In [19]:
from sklearn.neighbors import NearestNeighbors

# Fit model using cosine metric. 
# We set n_neighbors=3 because querying a point in the training set 
# will always return the point itself as the closest match (distance 0).
nn_model = NearestNeighbors(n_neighbors=3, metric='cosine')
nn_model.fit(X)

# Query the first document (using iloc[[0]] to maintain a 2D array shape)
distances, indices = nn_model.kneighbors(X.iloc[[0]])

# indices[0][1:] will contain the indices of the 2 actual nearest neighbors

In [21]:
indices[0][1:]

array([3, 1])

**Interview Tip:** Always mention the "self-match" trap in KNN when querying points that already exist in your training/index set. Mentioning that you need $K+1$ neighbors proves you have actually built this in practice.

### Q2: KNN at Production Scale
**Question:** You have 50 million TF-IDF vectors. You need to return nearest neighbors in <50ms. Why will standard `scikit-learn` KNN fail, and how do you fix it?

**Answer:**
*   **Why it fails:** Standard KNN is an exact, "brute-force" algorithm. It has $O(N \times D)$ inference time. To find neighbors, it must load all 50M vectors into memory and calculate the exact distance from the query to every single vector. This will take seconds or minutes, failing the 50ms SLA.
*   **The Solution:** We must switch from Exact Nearest Neighbors to **Approximate Nearest Neighbors (ANN)**. 
*   **How it works:** We use a library like **FAISS** (Facebook AI Similarity Search) or a Vector Database (Pinecone, Milvus, Qdrant). These tools pre-build indexes (using techniques like HNSW or IVF) that group similar vectors together. At inference time, they only search the relevant "neighborhood" of vectors, reducing search time to milliseconds at the cost of a tiny fraction of accuracy.

**Interview Tip:** If an interviewer asks about RAG (Retrieval-Augmented Generation) in this context, clarify that ANN/Vector Databases are the engine powering the "R" (Retrieval) in RAG!

### Q3: The Curse of Dimensionality
**Question:** Explain the "Curse of Dimensionality" in the context of KNN using Euclidean distance on a dataset with 5,000 features. 

**Answer:**
*   **The Problem:** KNN relies on distance metrics to find similar points. In highly dimensional spaces (like 5,000 features), the volume of the feature space expands exponentially, causing the data points to become incredibly sparse. 
*   **The Mathematical Breakdown:** As dimensions increase, the ratio of the distance to the nearest neighbor and the distance to the farthest neighbor converges to 1. In other words, all data points become roughly equidistant from each other.
*   **The Result:** Because every point is basically the same distance away, the concept of "closeness" or "nearest" breaks down entirely, and KNN's predictions become essentially random guesses.
*   **The Fix:** Dimensionality reduction (PCA, t-SNE) or feature selection must be applied before using KNN on high-dimensional dense data.

### Q4: The Bias-Variance Tradeoff in KNN
**Question:** How does the choice of $K$ ($K=1$ vs. $K=100$) impact the Bias-Variance tradeoff? Which would you use for a noisy dataset containing outliers?

**Ideal Answer:**
*   **$K=1$ (Low Bias, High Variance):** The model perfectly fits the training data (memorization), but becomes highly sensitive to noise. It behaves like changing your entire lifestyle based on a single conflicting opinion. This leads to severe overfitting.
*   **$K=100$ (High Bias, Low Variance):** The model takes an average/vote across a large neighborhood, which smooths out noise and outliers. It acts like consulting a panel of 100 experts before making a decision. However, if $K$ is too large, it risks underfitting by drawing too rigid of a boundary.
*   **For Noisy Data:** You must choose a larger $K$ (like $K=100$) to average out the impact of the outliers. 

**Interview Tip:** If the dataset is highly imbalanced (like fraud detection), be careful not to set $K$ *too* high. If $K$ is larger than the total number of minority class samples in a given region, the majority class will always win the vote, completely blinding the model to the minority class.

### Q5: Tabular Data Preprocessing for KNN
**Question:** You have a dataset with `Age` (18-80), `Annual_Income` ($20k-$250k), and `Credit_Tier` ('Poor', 'Fair', 'Good', 'Excellent'). How must you preprocess this for a KNN model using Euclidean distance, and why?

**Ideal Answer:**
*   **The Problem:** Euclidean distance calculates absolute geometric differences. Without preprocessing, the model will fail on the string data, and the massive magnitude of `Income` will completely drown out the `Age` feature. The model would basically only group people by income.
*   **Step 1: Ordinal Encoding (Credit_Tier):** Categorical strings must be converted to numbers. Because the tiers have a strict mathematical order, we use Ordinal Encoding (e.g., Poor=1, Fair=2, Good=3, Excellent=4) rather than One-Hot Encoding.
*   **Step 2: Log Transformation (Annual_Income) - *Optional but impressive*:** Income is usually highly right-skewed. Applying a log transform pulls extreme high-earners closer to the mean, preventing extreme outliers from skewing distances.
*   **Step 3: Feature Scaling (Age & Income):** We must apply `StandardScaler` (Z-score normalization) to all numeric features. This ensures both Age and Income have a mean of 0 and a variance of 1, allowing them to contribute equally to the distance calculation.

### Q6:
**The Question:**
Using scikit-learn, write a script that utilizes a ColumnTransformer to apply a StandardScaler to the numeric columns and an OrdinalEncoder to the Credit_Tier column (ensure the ordering is correct: Poor -> Fair -> Good -> Excellent).
Then, chain this transformer into a Pipeline with a KNeighborsClassifier ($K=3$) and fit the pipeline on X and y.

In [ ]:
# Data:
import pandas as pd

data = {
    'Age': [25, 45, 30, 50, 22],
    'Annual_Income': [35000, 120000, 50000, 210000, 20000],
    'Credit_Tier': ['Poor', 'Good', 'Fair', 'Excellent', 'Poor'],
    'Default': [1, 0, 1, 0, 1] # 1 = Defaulted, 0 = Paid
}
df = pd.DataFrame(data)

X = df.drop('Default', axis=1)
y = df['Default']

In [27]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer, OrdinalEncoder
from sklearn.neighbors import KNeighborsClassifier

# 1. Define Sub-Pipelines for each feature type
age_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])

income_pipeline = Pipeline(steps=[
    # log1p handles potential zeroes better than log
    ('log_transformer', FunctionTransformer(np.log1p)), 
    ('scaler', StandardScaler())
])

credit_pipeline = Pipeline(steps=[
    # Ensure strict ordering from lowest to highest
    ('encoder', OrdinalEncoder(categories=[['Poor', 'Fair', 'Good', 'Excellent']]))
])

# 2. Combine using ColumnTransformer
preprocessing = ColumnTransformer(transformers=[
    ('age', age_pipeline, ['Age']),
    ('income', income_pipeline, ['Annual_Income']),
    ('credit', credit_pipeline, ['Credit_Tier'])
])

# 3. Create Final Model Pipeline
model = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('classifier', KNeighborsClassifier(n_neighbors=3, p=2)) # p=2 is Euclidean
])

# Fit the entire pipeline
model.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](3,)","['Age','Annual_Income','Credit_Tier']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,3
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('age', ...), ('income', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns that

**Interview Tip:** Always use `np.log1p` instead of `np.log` in production pipelines to gracefully handle edge cases where a value might be exactly `0`, which would otherwise result in a `-inf` error.